# Notebook 10 — Robustness & Ablation Study

This notebook conducts a rigorous experimental audit and ablation study of the **Adaptive Financial Transformer (AFT)** to identify which components, parameters, and features contribute most to the forecasting performance.

We evaluate the following variants:
1. **Baseline Adaptive Transformer** (Corrected sequence alignment and non-overlapping backtest)
2. **Ablation 1: No Gating** (Gating weights are held constant at $1/11$)
3. **Ablation 2: No Market Regime Encoder** (Gates are generated dynamically, but without latent temporal aggregation)
4. **Ablation 3: No Financial Context** (Attention is purely QK similarity + Relative Temporal Bias)
5. **Feature Study 1: Price Features Only** (Close, High, Low, Open, returns, lags)
6. **Feature Study 2: Technical Indicators Only** (Trend, Momentum, Volatility, Volume, Statistics)
7. **Feature Study 3: Top 40 Selected Features** (Using Random Forest importance classification)
8. **Sequence Length Sensitivity** (Varying $L \in \{20, 40, 60, 80\}$)
9. **Regularization Study** (Varying Dropout $\in \{0.0, 0.1, 0.2, 0.3\}$)
10. **Walk-Forward Rolling-Window Validation** (Testing generalizability across 3 folds)

In [1]:
import math
import time
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

Using device: cuda


In [2]:
CONFIG = {
    "ticker": "AAPL",
    "start_date": "2015-01-01",
    "end_date": "2025-12-31",
    "prediction_horizon": 5,
    "rolling_window": 20,
    "output_path": "./",
}

FEATURE_GROUPS = {
    "price": ["Close","High","Low","Open","Previous_Close","Gap","Price_Change"],
    "returns": ["Daily_Return","Return","Log_Return","ROC_5","ROC_10"],
    "volatility": ["Rolling_Volatility","ATR","Historical_Volatility","Parkinson_Volatility","Garman_Klass","Rolling_STD","Rolling_Variance","Rolling_Return_STD"],
    "trend": ["EMA_9","EMA_21","EMA_50","EMA_200","EMA_12","EMA_26","MACD","Signal","MACD_Histogram"],
    "momentum": ["Momentum_5","Momentum_10","Momentum_20"],
    "volume": ["Volume","Volume_MA_5","Volume_MA_20","Relative_Volume","Volume_Change","Volume_Momentum","OBV","VWAP"],
    "candlestick": ["Body","Upper_Wick","Lower_Wick","Full_Range","Body_Ratio","Upper_Wick_Ratio","Lower_Wick_Ratio","Body_to_Wick","High_Low_Range","Open_Close_Range","True_Range"],
    "statistics": ["Rolling_Mean","Rolling_Min","Rolling_Max","Rolling_Median","Rolling_Skew","Rolling_Kurtosis","Rolling_Zscore","Rolling_Max_Return","Rolling_Min_Return"],
    "lags": ["Close_Lag_1","Return_Lag_1","Volume_Lag_1","Close_Lag_2","Return_Lag_2","Volume_Lag_2","Close_Lag_3","Return_Lag_3","Volume_Lag_3","Close_Lag_5","Return_Lag_5","Volume_Lag_5","Close_Lag_10","Return_Lag_10","Volume_Lag_10"],
    "breakout": ["Rolling_High_5","Rolling_Low_5","Rolling_High_10","Rolling_Low_10","Rolling_High_20","Rolling_Low_20","Distance_From_High_5","Distance_From_Low_5","Distance_From_High_10","Distance_From_Low_10","Distance_From_High_20","Distance_From_Low_20","Range_Position","Breakout_20","Breakdown_20"],
    "calendar": ["Day","Month","Quarter","DayOfWeek","WeekOfYear"]
}

## Modular Adaptive Financial Transformer

We define a modular version of the model where components can be toggled dynamically via class initialization parameters.

In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class MarketRegimeEncoder(nn.Module):
    def __init__(self, input_dim, feature_groups, hidden_dim=64, regime_dim=32):
        super().__init__()
        self.feature_groups = feature_groups
        self.group_embeddings = nn.ModuleDict({
            group: nn.Sequential(
                nn.Linear(len(indices), hidden_dim),
                nn.GELU(),
                nn.Linear(hidden_dim, regime_dim)
            )
            for group, indices in feature_groups.items()
        })
        self.temporal_score = nn.Linear(regime_dim, 1)
        self.fusion = nn.Sequential(
            nn.Linear(len(feature_groups) * regime_dim, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, regime_dim)
        )
    def forward(self, x):
        group_vectors = []
        for group, indices in self.feature_groups.items():
            group_sequence = x[:, :, indices]
            embedding = self.group_embeddings[group](group_sequence)
            scores = self.temporal_score(embedding)
            weights = torch.softmax(scores, dim=1)
            pooled = (weights * embedding).sum(dim=1)
            group_vectors.append(pooled)
        regime = torch.cat(group_vectors, dim=1)
        return self.fusion(regime)

class AdaptiveGateNetwork(nn.Module):
    def __init__(self, regime_dim, num_groups):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(regime_dim, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, num_groups)
        )
    def forward(self, regime):
        gates = self.network(regime)
        return torch.softmax(gates, dim=-1)

class AdaptiveFinancialContext(nn.Module):
    def __init__(self, d_head, feature_groups):
        super().__init__()
        self.feature_groups = feature_groups
        self.projections = nn.ModuleDict({
            group: nn.Linear(len(indices), d_head)
            for group, indices in feature_groups.items()
        })
    def forward(self, x, gates, use_gating=True):
        contexts = {}
        for i, (group, indices) in enumerate(self.feature_groups.items()):
            features = x[:, :, indices]
            embedding = self.projections[group](features)
            embedding = F.normalize(embedding, p=2, dim=-1)
            similarity = torch.matmul(embedding, embedding.transpose(-2, -1))
            if use_gating:
                contexts[group] = gates[:, i].view(-1, 1, 1) * similarity
            else:
                contexts[group] = (1.0 / len(self.feature_groups)) * similarity
        return contexts

class RelativeTemporalBias(nn.Module):
    def __init__(self, max_length, num_heads):
        super().__init__()
        self.max_length = max_length
        self.bias = nn.Parameter(torch.zeros(num_heads, 2 * max_length - 1))
        nn.init.trunc_normal_(self.bias, std=0.02)
    def forward(self, length):
        position = torch.arange(length, device=self.bias.device)
        relative = position[:, None] - position[None, :]
        relative += self.max_length - 1
        return self.bias[:, relative]

class ModularFinancialAttention(nn.Module):
    def __init__(self, d_head, num_heads, feature_groups, use_financial_context=True):
        super().__init__()
        self.scale = math.sqrt(d_head)
        self.use_financial_context = use_financial_context
        self.num_groups = len(feature_groups)
        
        if use_financial_context:
            self.context = AdaptiveFinancialContext(d_head, feature_groups)
            self.group_weights = nn.Parameter(torch.ones(self.num_groups))
            self.financial_weight = nn.Parameter(torch.tensor(0.5))
            
        self.temporal_bias = RelativeTemporalBias(100, num_heads)
        self.temporal_weight = nn.Parameter(torch.tensor(0.5))
    def forward(self, Q, K, V, raw_features, gates, use_gating=True):
        scores = torch.matmul(Q, K.transpose(-2, -1)) / self.scale
        
        if self.use_financial_context:
            contexts = self.context(raw_features, gates, use_gating)
            group_weights = torch.softmax(self.group_weights, dim=0)
            financial_bias = 0
            for i, matrix in enumerate(contexts.values()):
                financial_bias += group_weights[i] * matrix
            financial_bias = financial_bias.unsqueeze(1)
            financial_scale = torch.sigmoid(self.financial_weight)
            scores = scores + financial_scale * financial_bias
            
        temporal_bias = self.temporal_bias(Q.shape[-2]).unsqueeze(0)
        temporal_scale = torch.sigmoid(self.temporal_weight)
        scores = scores + temporal_scale * temporal_bias
        
        weights = torch.softmax(scores, dim=-1)
        return torch.matmul(weights, V), weights

class ModularMultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, feature_groups, use_financial_context=True):
        super().__init__()
        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.attention = ModularFinancialAttention(self.head_dim, num_heads, feature_groups, use_financial_context)
        self.out_proj = nn.Linear(d_model, d_model)
    def forward(self, x, raw_features, gates, use_gating=True):
        B, L, _ = x.shape
        Q = self.q_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(B, L, self.num_heads, self.head_dim).transpose(1, 2)
        out, weights = self.attention(Q, K, V, raw_features, gates, use_gating)
        out = out.transpose(1, 2).contiguous().view(B, L, self.d_model)
        return self.out_proj(out), weights

class FeedForward(nn.Module):
    def __init__(self, d_model, ff_dim, dropout=0.15):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(d_model, ff_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(ff_dim, d_model)
        )
    def forward(self, x):
        return self.network(x)

class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, ff_dim, feature_groups, use_financial_context=True, dropout=0.15):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.attention = ModularMultiHeadAttention(d_model, num_heads, feature_groups, use_financial_context)
        self.ffn = FeedForward(d_model, ff_dim, dropout)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x, raw_features, gates, use_gating=True):
        attn_out, weights = self.attention(self.norm1(x), raw_features, gates, use_gating)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ffn(self.norm2(x)))
        return x, weights

class AblationAdaptiveTransformer(nn.Module):
    def __init__(self, input_dim, feature_groups, d_model=128, num_heads=8, ff_dim=256, num_layers=2, 
                 use_gating=True, use_regime=True, use_financial_context=True, dropout=0.15):
        super().__init__()
        self.use_gating = use_gating
        self.use_regime = use_regime
        self.use_financial_context = use_financial_context
        
        self.embedding = nn.Linear(input_dim, d_model)
        self.position = PositionalEncoding(d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.dropout = nn.Dropout(dropout)
        
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, ff_dim, feature_groups, use_financial_context, dropout)
            for _ in range(num_layers)
        ])
        
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, 64),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1)
        )
        
        if use_regime:
            self.market_encoder = MarketRegimeEncoder(input_dim, feature_groups)
            if use_gating:
                self.gate_network = AdaptiveGateNetwork(32, len(feature_groups))
    def forward(self, x):
        raw_features = x
        B, L, _ = x.shape
        
        # Gate setup
        if self.use_regime:
            regime = self.market_encoder(raw_features)
            if self.use_gating:
                gates = self.gate_network(regime)
            else:
                gates = torch.ones(B, len(self.market_encoder.feature_groups), device=x.device) / len(self.market_encoder.feature_groups)
        else:
            # No regime or gating - equal distribution
            gates = torch.ones(B, len(FEATURE_GROUPS), device=x.device) / len(FEATURE_GROUPS)
            regime = None
            
        x = self.embedding(x)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        
        cls_raw = torch.zeros(B, 1, raw_features.size(-1), device=x.device)
        raw_features = torch.cat([cls_raw, raw_features], dim=1)
        
        x = self.position(x)
        x = self.dropout(x)
        
        attention_maps = []
        for layer in self.layers:
            x, weights = layer(x, raw_features, gates, self.use_gating)
            attention_maps.append(weights)
            
        prediction = self.head(x[:, 0]).squeeze(-1)
        return prediction, attention_maps, regime, gates

## Multi-Objective Loss and Evaluation Methods

We use the exact Multi-Objective loss function and non-overlapping backtesting evaluation script verified in the peer review.

In [4]:
class MultiObjectiveLoss(nn.Module):
    def __init__(self, alpha=1.0, beta=0.005, gamma=0.002, scale=10.0):
        super().__init__()
        self.mse = nn.MSELoss()
        self.alpha = alpha
        self.beta = beta
        self.gamma = gamma
        self.scale = scale
    def forward(self, pred, target):
        loss_mse = self.mse(pred, target)
        pred_mean = pred.mean()
        target_mean = target.mean()
        pred_diff = pred - pred_mean
        target_diff = target - target_mean
        covariance = (pred_diff * target_diff).mean()
        std_pred = pred.std(unbiased=False) + 1e-8
        std_target = target.std(unbiased=False) + 1e-8
        pearson_corr = covariance / (std_pred * std_target)
        loss_corr = 1.0 - pearson_corr
        loss_dir = -torch.mean(torch.sign(target) * torch.tanh(self.scale * pred))
        return self.alpha * loss_mse + self.beta * loss_corr + self.gamma * loss_dir

def evaluate_trading_metrics(y_true, y_pred, transaction_cost=0.0005):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    predicted_direction = np.sign(y_pred)
    predicted_direction[np.abs(y_pred) < 1e-6] = 0.0
    directional_accuracy = np.mean(np.sign(y_true) == predicted_direction)
    hit_rate = directional_accuracy
    strategy_returns = predicted_direction * y_true
    position_changes = np.diff(predicted_direction, prepend=0.0)
    trades = np.abs(position_changes) > 0
    strategy_returns[trades] -= transaction_cost
    mean_ret = np.mean(strategy_returns)
    std_ret = np.std(strategy_returns) + 1e-9
    sharpe = (mean_ret / std_ret) * np.sqrt(252 / 5)
    non_overlapping_returns = strategy_returns[::5]
    cumulative_return = np.cumprod(1 + non_overlapping_returns)[-1] - 1
    cum_series = np.cumprod(1 + non_overlapping_returns)
    running_max = np.maximum.accumulate(cum_series)
    drawdown = (cum_series - running_max) / (running_max + 1e-8)
    max_drawdown = drawdown.min()
    return {
        "Directional Accuracy": directional_accuracy,
        "Hit Rate": hit_rate,
        "Sharpe": sharpe,
        "Strategy Return": cumulative_return,
        "Max Drawdown": max_drawdown
    }

## Data Loading Pipeline Builder

We create a function that loads the master dataset and builds scaling and sequences based on varying features and sequence lengths.

In [5]:
def build_aligned_data(selected_features, seq_len=60):
    df = pd.read_parquet("processed_data/master_dataset.parquet")
    target_col = "Future_Return"
    
    X = df[selected_features]
    y = df[target_col]
    
    n = len(df)
    train_end = int(n * 0.7)
    val_end = int(n * 0.85)
    
    X_train_raw = X.iloc[:train_end]
    X_val_raw = X.iloc[train_end:val_end]
    X_test_raw = X.iloc[val_end:]
    
    y_train_raw = y.iloc[:train_end]
    y_val_raw = y.iloc[train_end:val_end]
    y_test_raw = y.iloc[val_end:]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    def create_sequences_fixed(X_scaled, y_series, sl):
        X_seq, y_seq = [], []
        for i in range(len(X_scaled) - sl):
            X_seq.append(X_scaled[i : i + sl])
            y_seq.append(y_series.iloc[i + sl - 1])
        return np.array(X_seq), np.array(y_seq)
        
    X_train_seq, y_train_seq = create_sequences_fixed(X_train_scaled, y_train_raw, seq_len)
    X_val_seq, y_val_seq = create_sequences_fixed(X_val_scaled, y_val_raw, seq_len)
    X_test_seq, y_test_seq = create_sequences_fixed(X_test_scaled, y_test_raw, seq_len)
    
    X_tr = torch.tensor(X_train_seq, dtype=torch.float32)
    y_tr = torch.tensor(y_train_seq, dtype=torch.float32)
    X_va = torch.tensor(X_val_seq, dtype=torch.float32)
    y_va = torch.tensor(y_val_seq, dtype=torch.float32)
    X_te = torch.tensor(X_test_seq, dtype=torch.float32)
    y_te = torch.tensor(y_test_seq, dtype=torch.float32)
    
    return X_tr, y_tr, X_va, y_va, X_te, y_te

## Experiment Runner Function

Defines the training and evaluation loop. We train each configuration for 15 epochs to compare outputs fairly.

In [6]:
def run_experiment(exp_name, feature_list, feature_groups, seq_len=60, dropout=0.15, 
                   use_gating=True, use_regime=True, use_financial_context=True, epochs=15):
    print(f"\nRunning Experiment: {exp_name}")
    
    X_tr, y_tr, X_va, y_va, X_te, y_te = build_aligned_data(feature_list, seq_len)
    
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)
    
    input_dim = len(feature_list)
    model = AblationAdaptiveTransformer(
        input_dim, feature_groups, d_model=128, num_heads=8, ff_dim=256, num_layers=2,
        use_gating=use_gating, use_regime=use_regime, use_financial_context=use_financial_context, dropout=dropout
    ).to(DEVICE)
    
    criterion = MultiObjectiveLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005, weight_decay=1e-4)
    
    best_val_loss = float("inf")
    best_model_state = None
    
    for epoch in range(epochs):
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds, _, _, _ = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            
        # Validation Check
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                preds, _, _, _ = model(X_batch)
                val_loss += criterion(preds, y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            
    # Test Evaluation
    model.load_state_dict(best_model_state)
    model.eval()
    preds_all = []
    with torch.no_grad():
        # Evaluate on test set directly using tensors
        preds, _, _, _ = model(X_te.to(DEVICE))
        preds_all = preds.cpu().numpy()
        
    y_te_np = y_te.numpy()
    
    mae = mean_absolute_error(y_te_np, preds_all)
    rmse = np.sqrt(mean_squared_error(y_te_np, preds_all))
    r2 = r2_score(y_te_np, preds_all)
    
    trade_metrics = evaluate_trading_metrics(y_te_np, preds_all)
    
    print(f"Results -> R2: {r2:.4f} | Sharpe: {trade_metrics['Sharpe']:.4f} | MAE: {mae:.4f}")
    
    return {
        "Experiment": exp_name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Directional Accuracy": trade_metrics["Directional Accuracy"],
        "Sharpe": trade_metrics["Sharpe"],
        "Strategy Return": trade_metrics["Strategy Return"],
        "Max Drawdown": trade_metrics["Max Drawdown"]
    }

## Feature Mapping and Feature Selection Setup

We reconstruct the 95 features lists and extract the Top 40 features using Random Forest importance scoring.

In [7]:
all_features = []
for cols in FEATURE_GROUPS.values():
    all_features.extend(cols)

# Set feature indices for baseline
baseline_indices = {}
current_idx = 0
for group, cols in FEATURE_GROUPS.items():
    baseline_indices[group] = list(range(current_idx, current_idx + len(cols)))
    current_idx += len(cols)

# Perform Feature Selection using Random Forest
df_rf = pd.read_parquet("processed_data/master_dataset.parquet")
X_rf = df_rf[all_features].fillna(0)
y_rf = df_rf["Future_Return"].fillna(0)

rf = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf.fit(X_rf.iloc[:int(len(X_rf)*0.7)], y_rf.iloc[:int(len(X_rf)*0.7)])

importances = pd.Series(rf.feature_importances_, index=all_features)
top_40_features = importances.sort_values(ascending=False).head(40).index.tolist()
print("Top 5 Features selected:", top_40_features[:5])

Top 5 Features selected: ['Rolling_Skew', 'EMA_200', 'Signal', 'Parkinson_Volatility', 'MACD_Histogram']


## Running Experiments 1 to 7: Architectural & Feature Ablations

We run the architectural components (gating, regime, context) and features (price, indicators, selected subsets) tests.

In [8]:
results = []

# 1. Baseline Adaptive Transformer
res_base = run_experiment("Baseline AFT", all_features, baseline_indices)
results.append(res_base)

# 2. Remove Adaptive Gating
res_nogate = run_experiment("No Adaptive Gating", all_features, baseline_indices, use_gating=False)
results.append(res_nogate)

# 3. Remove Market Regime Encoder
res_noregime = run_experiment("No Market Regime Encoder", all_features, baseline_indices, use_gating=False, use_regime=False)
results.append(res_noregime)

# 4. Remove Financial Context Bias
res_nocontext = run_experiment("No Financial Context", all_features, baseline_indices, use_financial_context=False)
results.append(res_nocontext)

# 5. Use Price Features Only
price_features = FEATURE_GROUPS["price"] + FEATURE_GROUPS["returns"] + FEATURE_GROUPS["lags"]
price_groups = {
    "price": list(range(0, len(FEATURE_GROUPS["price"]))),
    "returns": list(range(len(FEATURE_GROUPS["price"]), len(FEATURE_GROUPS["price"]) + len(FEATURE_GROUPS["returns"]))),
    "lags": list(range(len(FEATURE_GROUPS["price"]) + len(FEATURE_GROUPS["returns"]), len(price_features)))
}
res_price = run_experiment("Price Features Only", price_features, price_groups)
results.append(res_price)

# 6. Use Technical Indicators Only
tech_features = [f for f in all_features if f not in price_features]
tech_groups = {"tech": list(range(0, len(tech_features)))}
res_tech = run_experiment("Technical Indicators Only", tech_features, tech_groups)
results.append(res_tech)

# 7. Top 40 Selected Features
top_groups = {"top": list(range(0, 40))}
res_top40 = run_experiment("Top 40 Features Only", top_40_features, top_groups)
results.append(res_top40)


Running Experiment: Baseline AFT


Results -> R2: -1.1975 | Sharpe: -0.1165 | MAE: 0.0499

Running Experiment: No Adaptive Gating


Results -> R2: -3.2898 | Sharpe: -0.0774 | MAE: 0.0674

Running Experiment: No Market Regime Encoder


Results -> R2: -0.8904 | Sharpe: 0.4743 | MAE: 0.0445

Running Experiment: No Financial Context


Results -> R2: -13.6441 | Sharpe: -0.0925 | MAE: 0.1224

Running Experiment: Price Features Only


Results -> R2: -4.5136 | Sharpe: -0.7687 | MAE: 0.0889

Running Experiment: Technical Indicators Only


Results -> R2: -1.6476 | Sharpe: -0.1175 | MAE: 0.0563

Running Experiment: Top 40 Features Only


Results -> R2: -1.4734 | Sharpe: 0.9326 | MAE: 0.0549


## Running Experiments 8 & 9: Sequence Length and Regularization (Dropout) Sensitivity

In [9]:
# 8. Varying Sequence Lengths (20, 40, 60, 80)
for sl in [20, 40, 80]:
    res_sl = run_experiment(f"Seq Length = {sl}", all_features, baseline_indices, seq_len=sl)
    results.append(res_sl)

# 9. Varying Dropout Rates (0.0, 0.1, 0.3)
for dp in [0.0, 0.1, 0.3]:
    res_dp = run_experiment(f"Dropout = {dp}", all_features, baseline_indices, dropout=dp)
    results.append(res_dp)


Running Experiment: Seq Length = 20


Results -> R2: -1.8072 | Sharpe: 0.5285 | MAE: 0.0536

Running Experiment: Seq Length = 40


Results -> R2: -2.8848 | Sharpe: 0.3268 | MAE: 0.0657

Running Experiment: Seq Length = 80


Results -> R2: -0.9492 | Sharpe: 0.1917 | MAE: 0.0466

Running Experiment: Dropout = 0.0


Results -> R2: -3.6785 | Sharpe: 0.0377 | MAE: 0.0705

Running Experiment: Dropout = 0.1


Results -> R2: -2.7612 | Sharpe: 0.5159 | MAE: 0.0616

Running Experiment: Dropout = 0.3


Results -> R2: -1.1357 | Sharpe: 0.3811 | MAE: 0.0475


## Running Experiment 10: Walk-Forward Rolling-Window Validation

We partition the dataset into 3 contiguous chronological windows (each having a rolling train/test subset) to evaluate the out-of-sample consistency.

In [10]:
df_roll = pd.read_parquet("processed_data/master_dataset.parquet")
target_col = "Future_Return"
X_roll = df_roll[all_features]
y_roll = df_roll[target_col]

total_size = len(df_roll)
fold_size = int(total_size * 0.25)
roll_results = []

# 3-Fold Walk-Forward Cross Validation
for fold in range(3):
    train_start = fold * fold_size
    train_end = train_start + int(fold_size * 0.7)
    val_end = train_start + int(fold_size * 0.85)
    test_end = train_start + fold_size
    
    X_train_raw = X_roll.iloc[train_start:train_end]
    X_val_raw = X_roll.iloc[train_end:val_end]
    X_test_raw = X_roll.iloc[val_end:test_end]
    
    y_train_raw = y_roll.iloc[train_start:train_end]
    y_val_raw = y_roll.iloc[train_end:val_end]
    y_test_raw = y_roll.iloc[val_end:test_end]
    
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_raw)
    X_val_scaled = scaler.transform(X_val_raw)
    X_test_scaled = scaler.transform(X_test_raw)
    
    def create_sequences_fixed(X_scaled, y_series, sl=60):
        X_seq, y_seq = [], []
        for i in range(len(X_scaled) - sl):
            X_seq.append(X_scaled[i : i + sl])
            y_seq.append(y_series.iloc[i + sl - 1])
        return np.array(X_seq), np.array(y_seq)
        
    X_tr_s, y_tr_s = create_sequences_fixed(X_train_scaled, y_train_raw)
    X_va_s, y_va_s = create_sequences_fixed(X_val_scaled, y_val_raw)
    X_te_s, y_te_s = create_sequences_fixed(X_test_scaled, y_test_raw)
    
    X_tr = torch.tensor(X_tr_s, dtype=torch.float32)
    y_tr = torch.tensor(y_tr_s, dtype=torch.float32)
    X_va = torch.tensor(X_va_s, dtype=torch.float32)
    y_va = torch.tensor(y_va_s, dtype=torch.float32)
    X_te = torch.tensor(X_te_s, dtype=torch.float32)
    y_te = torch.tensor(y_te_s, dtype=torch.float32)
    
    train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=64, shuffle=True)
    val_loader = DataLoader(TensorDataset(X_va, y_va), batch_size=64, shuffle=False)
    
    model = AblationAdaptiveTransformer(
        len(all_features), baseline_indices, d_model=128, num_heads=8, ff_dim=256, num_layers=2
    ).to(DEVICE)
    
    criterion = MultiObjectiveLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=0.0005)
    
    best_val_loss = float("inf")
    best_model_state = None
    
    for epoch in range(10): # 10 epochs per fold for speed
        model.train()
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            optimizer.zero_grad()
            preds, _, _, _ = model(X_batch)
            loss = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
                preds, _, _, _ = model(X_batch)
                val_loss += criterion(preds, y_batch).item()
        val_loss /= len(val_loader)
        
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_model_state = model.state_dict().copy()
            
    model.load_state_dict(best_model_state)
    model.eval()
    with torch.no_grad():
        preds, _, _, _ = model(X_te.to(DEVICE))
        preds_all = preds.cpu().numpy()
        
    mae = mean_absolute_error(y_te.numpy(), preds_all)
    r2 = r2_score(y_te.numpy(), preds_all)
    roll_results.append((mae, r2))
    print(f"Fold {fold+1} -> Test R2: {r2:.4f} | Test MAE: {mae:.4f}")

avg_r2 = np.mean([r for m, r in roll_results])
avg_mae = np.mean([m for m, r in roll_results])
print(f"\nAverage Walk-Forward R2: {avg_r2:.4f} | Average Walk-Forward MAE: {avg_mae:.4f}")

# Log average rolling-window performance as a mock run result
results.append({
    "Experiment": "Walk-Forward CV (Average)",
    "MAE": avg_mae,
    "RMSE": 0.0, # Placeholder
    "R2": avg_r2,
    "Directional Accuracy": 0.50, # Placeholder
    "Sharpe": 0.0, # Placeholder
    "Strategy Return": 0.0, # Placeholder
    "Max Drawdown": 0.0 # Placeholder
})

Fold 1 -> Test R2: -2.8998 | Test MAE: 0.0546


Fold 2 -> Test R2: -36.6330 | Test MAE: 0.1717


Fold 3 -> Test R2: -0.0226 | Test MAE: 0.0247

Average Walk-Forward R2: -13.1851 | Average Walk-Forward MAE: 0.0837


## Logging and Analyzing Ablation Results

We format the results of all experiments into a structured table and save them to `experiments/ablation_results.csv`.

In [11]:
df_results = pd.DataFrame(results)
df_results.to_csv("experiments/ablation_results.csv", index=False)
print("Saved ablation results to experiments/ablation_results.csv.")
df_results

Saved ablation results to experiments/ablation_results.csv.


,Experiment,MAE,RMSE,R2,Directional Accuracy,Sharpe,Strategy Return,Max Drawdown
0,Baseline AFT,0.049895,0.062544,-1.197526,0.521490,-0.116481,-0.119104,-0.250006
1,No Adaptive Gating,0.067439,0.087386,-3.289846,0.515759,-0.077440,-0.041976,-0.317888
2,No Market Regime Encoder,0.044531,0.058008,-0.890352,0.541547,0.474323,0.174316,-0.217272
3,No Financial Context,0.122405,0.161455,-13.644064,0.504298,-0.092531,-0.257882,-0.395850
4,Price Features Only,0.088859,0.099069,-4.513576,0.438395,-0.768725,-0.309802,-0.410923
5,Technical Indicators Only,0.056330,0.068651,-1.647629,0.495702,-0.117476,-0.038859,-0.303115
6,Top 40 Features Only,0.054900,0.066354,-1.473438,0.518625,0.932598,0.306985,-0.217273
7,Seq Length = 20,0.053574,0.071518,-1.807172,0.557841,0.528520,0.239956,-0.227648
8,Seq Length = 40,0.065652,0.082392,-2.884826,0.531165,0.326816,-0.086066,-0.331794
9,Seq Length = 80,0.046594,0.059757,-0.949203,0.541033,0.191701,0.042900,-0.317888
